## tl;dr

A literal unlimited fixed-price plan cannot guarantee a profit because a single account can create unbounded OpenAI cost. The launch recommendation is **EUR 19.99/month including an assumed 21% Latvian VAT**, sold as high-use personal access with a clearly disclosed fair-use policy. The planning model leaves about EUR 15.97 after VAT and a standard EEA Stripe card fee, before AI and fixed infrastructure.

## Context & Methods

Decision: choose a safe launch price for WhatNow? Pro and identify the controls required before payments are enabled.

### Key Assumptions

- GPT-5.6 Luna: USD 1.00 / 1M input tokens, USD 0.10 / 1M cached input tokens, USD 6.00 / 1M output tokens.
- EUR/USD: 1 EUR = 1.1467 USD (ECB reference rate, 17 July 2026), so 1 USD = about EUR 0.8721.
- Stripe standard EEA card fee: 1.5% + EUR 0.25 per successful payment.
- Latvian standard VAT assumption: 21%. Actual tax treatment and registration obligations must be confirmed before launch.
- Scenarios are planning assumptions, not observed production usage. The application does not yet retain per-analysis token cost; this preparation adds that telemetry.

In [1]:
from dataclasses import dataclass

USD_PER_EUR = 1.1467
EUR_PER_USD = 1 / USD_PER_EUR
INPUT_USD_PER_M = 1.00
OUTPUT_USD_PER_M = 6.00
VAT_RATE = 0.21
STRIPE_RATE = 0.015
STRIPE_FIXED_EUR = 0.25

@dataclass(frozen=True)
class Scenario:
    name: str
    input_tokens: int
    output_tokens: int

def ai_cost_eur(scenario: Scenario) -> float:
    usd = scenario.input_tokens / 1_000_000 * INPUT_USD_PER_M + scenario.output_tokens / 1_000_000 * OUTPUT_USD_PER_M
    return usd * EUR_PER_USD

def contribution_before_ai(gross_price_eur: float) -> float:
    net_of_vat = gross_price_eur / (1 + VAT_RATE)
    stripe_fee = gross_price_eur * STRIPE_RATE + STRIPE_FIXED_EUR
    return net_of_vat - stripe_fee

scenarios = [
    Scenario('Light text', 4_000, 1_000),
    Scenario('Typical document', 12_000, 1_500),
    Scenario('Heavy PDF', 50_000, 2_500),
    Scenario('Very large request', 250_000, 3_500),
]
[(s.name, round(ai_cost_eur(s), 4)) for s in scenarios]

[('Light text', 0.0087), ('Typical document', 0.0183), ('Heavy PDF', 0.0567), ('Very large request', 0.2363)]

## Data

The model uses official provider prices and four bounded token scenarios. No personal document content, user data, or production events are included.

In [2]:
price_options = [9.99, 14.99, 19.99, 24.99]
price_table = []
for price in price_options:
    available = contribution_before_ai(price)
    price_table.append({
        'gross_price_eur': price,
        'after_vat_and_stripe_eur': round(available, 2),
        'typical_analyses_to_zero_margin': round(available / ai_cost_eur(scenarios[1])),
        'heavy_analyses_to_zero_margin': round(available / ai_cost_eur(scenarios[2])),
    })
price_table

[{'gross_price_eur': 9.99, 'after_vat_and_stripe_eur': 7.86, 'typical_analyses_to_zero_margin': 429, 'heavy_analyses_to_zero_margin': 139}, {'gross_price_eur': 14.99, 'after_vat_and_stripe_eur': 11.91, 'typical_analyses_to_zero_margin': 651, 'heavy_analyses_to_zero_margin': 210}, {'gross_price_eur': 19.99, 'after_vat_and_stripe_eur': 15.97, 'typical_analyses_to_zero_margin': 872, 'heavy_analyses_to_zero_margin': 282}, {'gross_price_eur': 24.99, 'after_vat_and_stripe_eur': 20.03, 'typical_analyses_to_zero_margin': 1094, 'heavy_analyses_to_zero_margin': 353}]

## Results

At EUR 19.99 gross, the planning contribution before AI is about EUR 15.97. One typical analysis costs about EUR 0.018, while a 50k-input-token PDF costs about EUR 0.057. The price therefore supports ordinary personal usage with room for support and fixed infrastructure, but it cannot cover an unbounded number of large requests.

In [3]:
recommended_price = 19.99
available = contribution_before_ai(recommended_price)
for monthly_analyses in (100, 300, 500):
    typical_cost = monthly_analyses * ai_cost_eur(scenarios[1])
    print(monthly_analyses, round(typical_cost, 2), round(available - typical_cost, 2))


100 1.83 14.14
300 5.49 10.48
500 9.16 6.81


## Takeaways

1. Launch price recommendation: EUR 19.99/month including VAT, subject to tax confirmation.
2. Do not promise literal unlimited usage. Use clear high-use/fair-use wording and publish the safety thresholds before checkout.
3. Keep per-document size/page/token ceilings, account authentication, CAPTCHA, account limits, and a global budget circuit breaker.
4. Collect token counts and estimated model cost without retaining document content in cost telemetry. Reprice after at least 100 successful real analyses or 30 days, whichever comes later.
5. Do not activate Stripe or paid entitlements until the owner approves the price, fair-use terms, refund/cancellation flow, and legal/tax readiness.